<a href="https://colab.research.google.com/github/bsheese/387/blob/main/20_how_networks_learn/20_0_PyTorch_Basics/20_0_1_Tensors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How Networks Learn: Part 1
## Tensors

---

## What This Notebook Is About

In 19_1 you worked with the Pets model's outputs as NumPy arrays: you compared arrays, selected rows with masks, and computed scores with `@`. The model itself, in 19_0, did all of its work on PyTorch **tensors**. A tensor is a NumPy-style array with two additions that deep learning depends on:

1. **It can live on a GPU.** The same code runs on a CPU or a GPU, depending on where the tensors are.
2. **It can keep track of how it was computed.** PyTorch uses that record to work out how each weight should change during training. That is the subject of 20_0_2.

This notebook is about tensors as containers for numbers. Most of what you know about NumPy arrays carries over unchanged, so the notebook concentrates on the places where tensors differ, and on **shapes**, which cause more errors in deep learning code than anything else.

**What you will learn:**
1. How to make tensors, and their four main attributes
2. How to move between NumPy arrays and tensors, and redo 19_1_3's matrix multiplication in PyTorch
3. The four places where tensors behave differently from arrays
4. How to read and change shapes, including the extra "batch" dimension that models expect
5. How code moves tensors to a GPU
6. What `requires_grad=True` does (a first look)

In [ ]:
import os
import urllib.request

import numpy as np
import torch

DATA_URL = 'https://raw.githubusercontent.com/bsheese/387/main/data/'

def fetch(name):
    # Download a file from the course's data folder, unless it is already here.
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATA_URL + name, name)
    return name

torch.manual_seed(0)
print('PyTorch version:', torch.__version__)

---

## Section 1: Making Tensors

`torch.tensor` makes a tensor from a list, the way `np.array` makes an array. A list of numbers gives a one-dimensional tensor and a list of lists gives a two-dimensional one.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
print(a)

b = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
print(b)

Every tensor has four attributes you will check again and again:

- **`.shape`**: the size along each dimension. Here (2, 3): 2 rows and 3 columns.
- **`.dtype`**: the type of number stored. PyTorch's default for decimals is `float32`.
- **`.ndim`**: the number of dimensions, the same as the length of the shape.
- **`.device`**: where the tensor is stored, `cpu` or a GPU such as `cuda:0`.

In [ ]:
print('shape :', b.shape)
print('dtype :', b.dtype)
print('ndim  :', b.ndim)
print('device:', b.device)
print('number of elements:', b.numel())

The dtype is one place where PyTorch and NumPy differ. NumPy stores decimals as `float64` by default and PyTorch uses `float32`, which takes half the memory and runs faster on GPUs. The extra precision of `float64` almost never matters for training a network.

The same shortcuts for making arrays exist for tensors. You will see them when weights are first created and when we need a tensor of a given shape to experiment with.

In [ ]:
print(torch.zeros(2, 3))
print(torch.ones(2, 3))
print(torch.rand(2, 3))            # random numbers between 0 and 1
print(torch.arange(0, 10, 2))      # like range(0, 10, 2)
print(torch.linspace(0, 1, 5))     # 5 evenly spaced numbers from 0 to 1

---

## Section 2: NumPy and PyTorch Together

Data often starts as a NumPy array, and results often need to go back to NumPy for plotting or for scikit-learn. `torch.tensor(array)` copies an array into a new tensor, and `.numpy()` turns a tensor back into an array.

Here are the Pets features from 19_1, which were saved as a NumPy array, and the model's last layer, which was saved as tensors:

In [ ]:
saved = np.load(fetch('pets4_features.npz'))
features_np = saved['features']
print('as an array: ', type(features_np).__name__, features_np.shape, features_np.dtype)

features = torch.tensor(features_np)
print('as a tensor: ', type(features).__name__, features.shape, features.dtype)

head = torch.load(fetch('pets4_head.pt'))
W, bias = head['weight'], head['bias']
print('W:', W.shape, '  bias:', bias.shape)

In 19_1_3 we computed every photo's four scores with NumPy as `features @ W.T + bias`. The same line works on tensors, and it gives a tensor of the same shape:

In [ ]:
scores = features @ W.T + bias
print('scores:', scores.shape)
print('first photo:', scores[0])

Those are the scores from 19_1_3 (2.44, −0.87, −4.38, −1.66). Indexing, slicing, arithmetic, `@`, `.sum()`, `.mean()`, `.T` and `.argmax()` all work on tensors the way they do on arrays. For example, the predicted breed of every photo:

In [ ]:
BREEDS = ['Birman', 'Ragdoll', 'basset_hound', 'beagle']
predicted = scores.argmax(dim=1)          # the column with the highest score, for each row
print(predicted[:5])
print([BREEDS[i] for i in predicted[:5]])

PyTorch calls the axis argument `dim` where NumPy calls it `axis`. `dim=1` means "across the columns", so each row gets one answer.

| Operation | NumPy | PyTorch |
|---|---|---|
| From a list | `np.array([1, 2, 3])` | `torch.tensor([1, 2, 3])` |
| Zeros | `np.zeros((m, n))` | `torch.zeros(m, n)` |
| One element / one column | `a[0, 1]`, `a[:, 1]` | `t[0, 1]`, `t[:, 1]` |
| Matrix multiply | `a @ b` | `t @ u` |
| Sum, mean, largest | `a.sum()`, `a.mean()`, `a.max()` | `t.sum()`, `t.mean()`, `t.max()` |
| Position of the largest, per row | `a.argmax(axis=1)` | `t.argmax(dim=1)` |
| Transpose | `a.T` | `t.T` |
| Join along an existing axis | `np.concatenate([a, b])` | `torch.cat([t, u])` |
| Back and forth | `t.numpy()` | `torch.tensor(a)` |

---

## Section 3: Four Differences That Cause Bugs

**1. Totals and averages are still tensors.** `.sum()` on a NumPy array gives a plain number. On a tensor it gives a tensor with no dimensions. `.item()` turns a one-element tensor into a Python number, which is what you want for printing or for storing a loss in a list.

In [ ]:
total = torch.tensor([1.0, 2.0, 3.0]).sum()
print(total, type(total).__name__)
print(total.item(), type(total.item()).__name__)

**2. Methods that end in an underscore change the tensor in place.** `x.add(10)` returns a new tensor and leaves `x` alone. `x.add_(10)` changes `x` itself. You will see `zero_()` in the next notebook, where it resets a tensor to zeros.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])
y = x.add(10)
print('after x.add(10): x =', x, '  y =', y)
x.add_(10)
print('after x.add_(10): x =', x)

**3. Converting from NumPy keeps NumPy's dtype.** An array of `float64` becomes a tensor of `float64`, and layers made by PyTorch expect `float32`. Mixing them gives an error that names the two dtypes. Passing `dtype=torch.float32` when you convert avoids it.

In [ ]:
arr = np.array([1.5, 2.5, 3.5])
print(torch.tensor(arr).dtype)
print(torch.tensor(arr, dtype=torch.float32).dtype)

layer = torch.nn.Linear(3, 1)          # a layer of one neuron with 3 weights, made by PyTorch
try:
    layer(torch.tensor(arr))
except RuntimeError as e:
    print('RuntimeError:', e)
print(layer(torch.tensor(arr, dtype=torch.float32)))

**4. `.numpy()` only works on tensors that are on the CPU and aren't tracking gradients.** For a tensor that is tracking gradients (Section 6), use `.detach().numpy()`; for a tensor on a GPU, add `.cpu()` first. The error messages say which one is needed.

---

## Section 4: Shapes

Most errors in deep learning code are shape errors, so it pays to be able to read shapes and change them. The shape rule for `@` from 19_1_3 applies to tensors unchanged: the inner sizes must match.

In [ ]:
print(features.shape, '@', W.T.shape, '->', (features @ W.T).shape)
try:
    features @ W
except RuntimeError as e:
    print('RuntimeError:', e)

PyTorch's message names the two shapes that didn't fit, "800x512 and 4x512". When you see an error like that, find the two shapes in it and ask which inner sizes were supposed to match.

### Changing a shape

`.reshape` rearranges the same numbers into a different shape. The total number of elements can't change, and `-1` means "whatever size makes the total work out".

In [ ]:
t = torch.arange(12)
print(t.shape)
print(t.reshape(3, 4))
print(t.reshape(2, -1).shape)    # -1 is worked out as 6

### The batch dimension

Models work on **batches**: several examples processed together, stacked along a first dimension. So every layer expects that first dimension, even when you only have one example.

Photos have their own shape. A color photo of 224 × 224 pixels, prepared for ResNet-18, is a tensor of shape (3, 224, 224): 3 color channels (red, green and blue), each a 224 × 224 grid of numbers. A batch of 32 such photos has shape (32, 3, 224, 224). That is the shape of `x` in each step of 19_0_1's training loop.

To pass a single photo to the model, it needs a batch dimension of size 1. That's what the `.unsqueeze(0)` in 19_0_1's `predict` function did: it adds a new dimension of size 1 in position 0.

In [ ]:
photo = torch.rand(3, 224, 224)       # stands in for one prepared photo
print('one photo:      ', photo.shape)
batch = photo.unsqueeze(0)
print('a batch of one: ', batch.shape)
print('back again:     ', batch.squeeze(0).shape)

thirty_two = torch.rand(32, 3, 224, 224)
print('a batch of 32:  ', thirty_two.shape, f'({thirty_two.numel():,} numbers)')

`.squeeze(0)` removes a dimension of size 1, undoing `.unsqueeze(0)`. A batch of 32 photos holds about 4.8 million numbers, which is one reason training goes much faster on hardware built to process numbers in parallel.

---

## Section 5: CPU or GPU

Every tensor lives on a **device**. Tensors that are combined in one calculation must be on the same device, so code usually decides on a device once and then moves the model and every batch of data there with `.to(device)`. You saw this pattern in 19_0_1:

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

x = torch.tensor([1.0, 2.0, 3.0]).to(device)
print(x.device)

In this notebook you are probably on the CPU, which is fine: the tensors here are small. The point of writing `.to(device)` everywhere is that the same notebook then runs on a GPU without any changes. Forgetting to move one tensor gives an error that mentions two devices, such as "Expected all tensors to be on the same device".

---

## Section 6: A First Look at `requires_grad`

The one thing tensors can do that arrays can't is remember how they were computed. Setting `requires_grad=True` on a tensor tells PyTorch to keep a record of every calculation that uses it.

In [ ]:
plain = torch.tensor([2.0, 3.0])
tracked = torch.tensor([2.0, 3.0], requires_grad=True)

y_plain = plain ** 2 + 3 * plain
y_tracked = tracked ** 2 + 3 * tracked

print(y_plain)
print(y_tracked)
print('y_tracked.grad_fn:', y_tracked.grad_fn)

The values are the same. The tracked result also carries `grad_fn`, a note of the last calculation that produced it (an addition), which links back to the calculations before it. During training, the weights of a model are tracked like this. The next notebook shows what PyTorch does with the record: it works out, for every weight at once, which way the weight should move to make the loss smaller.

---

## Summary

- A **tensor** is PyTorch's version of a NumPy array. It has a `.shape`, a `.dtype` (`float32` by default), an `.ndim` and a `.device`. Indexing, arithmetic, `@`, `.sum()`, `.argmax()` and `.T` work as they do in NumPy.
- `torch.tensor(array)` and `.numpy()` move data between NumPy and PyTorch. The Pets scores from 19_1_3 come out the same either way.
- Four differences from NumPy: totals are still tensors (use `.item()`); methods ending in `_` change a tensor in place; converting from NumPy keeps `float64` unless you ask for `float32`; `.numpy()` needs a CPU tensor that isn't tracking gradients.
- **Shapes**: the inner sizes in `@` must match; `.reshape` rearranges elements; models expect a **batch** dimension first, so one photo of shape (3, 224, 224) becomes (1, 3, 224, 224) with `.unsqueeze(0)`.
- `.to(device)` moves tensors to a GPU when there is one; tensors in one calculation must be on the same device.
- `requires_grad=True` makes PyTorch record how a tensor is used, which is what training relies on.

**Next:** 20_0_2 asks how a model knows which way to change each weight, first by nudging a weight and watching the loss, and then with PyTorch's much faster way of getting the same answer.